# Consensus panel — the human population vs DG3

MIT1003 records about fifteen subjects per image. Comparing DG3 to *one* human is
misleading: there is no single ground-truth scanpath, there is a population. Each
stimulus gets a three-panel figure:

1. **A.** Every human scanpath, one colour per subject.
2. **B.** Pixels where ≥75% (light) or ≥90% (dark, contoured) of subjects fixated.
3. **C.** DG3's probability map conditioned on one synthetic fixation at the image
   centre. This notebook uses the plain `get_mit1003` loader, which drops the
   recorded central start, so panel C's starting point is constructed rather than
   read from the data.

Colours use lightness ordering (light → dark) for colourblind safety.

A fourth panel drew a scanpath sampled from DG3. It was dropped: nothing in the
thesis is measured in the sampling mode, so the panel showed a mode no number
comes from.

The figure is built by `tez_deepgaze.consensus_panel.build_consensus_panel`, the
same function `scripts/demo_consensus_panel.py` uses, so the last cell reproduces
the six stacks ch04 includes rather than re-drawing them a second way.

## Setup
Finds the repo root, puts `src/` on the path, and imports the shared builder.

In [ ]:
import sys
from pathlib import Path

# Find the repo root (the folder with pyproject.toml) from wherever this runs.
REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

import matplotlib.pyplot as plt
import deepgaze_pytorch
import pysaliency

from tez_deepgaze.consensus_panel import build_consensus_panel, DEFAULT_OUT
from tez_deepgaze.human_scanpaths import CONSENSUS_RADIUS_PX
from tez_deepgaze.device import pick_device, to_device
from tez_deepgaze.paths import DATA_ROOT

print("repo:", REPO)
print("data:", DATA_ROOT)

## Load the model and data (once)

Loads MIT1003 and the pretrained DeepGaze III. Sampling is inference only, so
this runs on CPU / Apple MPS / CUDA — whatever `pick_device` finds. The first run
downloads the DG3 weights to the torch cache; later runs reuse them.

In [ ]:
device = pick_device()
print("device:", device)
stimuli, fixations = pysaliency.get_mit1003(location=str(DATA_ROOT))
model = to_device(deepgaze_pytorch.DeepGazeIII(pretrained=True), device).eval()
print("loaded DeepGaze III and", len(stimuli.stimuli), "MIT1003 stimuli")

## Look at one stimulus

Edit the parameters and re-run. `build_consensus_panel` returns `(fig, summary)`;
here we display the figure inline and do **not** save it.

In [ ]:
STIM_IDX = 91    # MIT1003 stimulus index (0..1002)
RADIUS   = CONSENSUS_RADIUS_PX   # 70 px = 2° at MIT1003 viewing geometry; override to explore

fig, summary = build_consensus_panel(
    STIM_IDX, stimuli=stimuli, fixations=fixations, model=model, device=device,
    radius=RADIUS,   # out_dir omitted → not saved
)
print(f"stim {STIM_IDX}: {summary['n_subjects']} subjects, "
      f"≥75% area {summary['consensus_area_pct']['th75']:.1f}%")
fig

## Regenerate the six stacks ch04 includes

ch04's "extreme stimuli by information gain" figure stacks six of these panels:
the three highest and the three lowest stimuli by per-image IG. They differ from
the exploration panel above in two ways, both set below — each title carries that
stimulus's IG, NSS and AUC read from the per-image diagnostic, and only the first
carries the A/B/C panel headings, because the figure stacks them.

This is the same call `scripts/demo_consensus_panel.py` makes:

```bash
.venv/bin/python scripts/demo_consensus_panel.py --stim-indices 128 704 528 750 639 489 \
    --metrics-from results/per_image_diagnostic_initial_1003/diagnostic.json \
    --titles-first-only --page-frac 0.5
```

Output goes to `results/consensus_panels/` (PNG plus a sidecar JSON with the
scanpaths and consensus numbers). The other panels in that directory are
exploration output and are not included by any chapter.

In [ ]:
import json

# The three highest and three lowest stimuli by per-image IG, in the order ch04
# stacks them. Read them from the diagnostic rather than hardcoding, so the set
# follows the artefact if it is ever re-evaluated.
DIAGNOSTIC = REPO / "results/per_image_diagnostic_initial_1003/diagnostic.json"

rows = json.loads(DIAGNOSTIC.read_text())["rows"]
metrics = {r["stim_idx"]: r["dg3"] for r in rows}
by_ig = sorted(metrics, key=lambda s: metrics[s]["ig_bits"])
THESIS_STIMS = by_ig[:-4:-1] + by_ig[:3]      # top three descending, bottom three ascending
print("ch04 stack:", THESIS_STIMS)

for k, sidx in enumerate(THESIS_STIMS):
    m = metrics[sidx]
    suffix = (f"   ·   IG {m['ig_bits']:.2f} bits   ·   NSS {m['nss']:.1f}"
              f"   ·   AUC {m['auc']:.3f}")
    fig, summary = build_consensus_panel(
        sidx, stimuli=stimuli, fixations=fixations, model=model, device=device,
        radius=RADIUS, out_dir=DEFAULT_OUT, title_suffix=suffix,
        panel_titles=(k == 0), page_frac=0.5,
    )
    plt.close(fig)   # batch: close so the notebook does not render all six
    print(f"stim {sidx:4d}  {summary['n_subjects']:2d} subjects → {summary['out_png']}")

print("\ndone — figures written to", DEFAULT_OUT)

## Try this

- Compare an "easy" image (77 — clear central target) with a "hard" one
  (522 — texture, no single target). Notice how the consensus area shrinks.
- Raise `RADIUS` to 80–100 px and watch the consensus blob grow.